In [1]:
import vertexai
from vertexai.generative_models import GenerativeModel, ChatSession, HarmCategory, HarmBlockThreshold
import json

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.aiplatform_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.aiplatform_v1beta1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1beta1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_supp

In [2]:
PROJECT_ID = "spring-2026-lifejacket"
REGION = "us-central1"
vertexai.init(project=PROJECT_ID, location=REGION)

In [3]:
SCENARIO = "BOTH"  # Scenario A
# SCENARIO = "IMAGE_ONLY" # Scenario B
#SCENARIO = "TEXT_ONLY" # Scenario C

In [4]:
#mock CV Data input
if SCENARIO == "TEXT_ONLY":
    cv_data = None 
else:
    cv_data = {
        "species": "California Sea Lion",
        "confidence": 0.92,
        "visual_status": "breathing but eyes closed",
        "apparent_injury": "none"
    }

In [5]:
#System Prompt
system_prompt = f"""
You are the LifeJacket Emergency Dispatcher. 

Current CV Input: {json.dumps(cv_data) if cv_data else "No image provided. Rely on user description."}

LOGIC:
- If CV Data is present: Acknowledge it (e.g., "I see a sea lion in the photo") and ask for anything to add.
- If CV Data is NULL: Ask the user to describe the animal's size, shape, and features for identification.
- Always: Give safety advice (stay 50ft back) and ask only ONE question at a time.
- Extraction Goal: Identify [species, health_status, location, urgency].
"""

# Model Generation Configur 
generation_config = {"temperature": 0.3, "max_output_tokens": 800}

#Initialize Generative Model and Chat Session using API
model = GenerativeModel("gemini-2.5-flash", generation_config=generation_config)
chat = model.start_chat()

/opt/conda/lib/python3.10/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [6]:
#Utility Function for GPS Mocking
def get_device_location():
    """Simulates mobile GPS API call"""
    return {
        "lat": 41.8818, 
        "lon": -87.6231,
        "address": "North Water Street, Chicago, IL"
    }

**Main Interaction Loop (3 Triggers)**
Three Triggers

In [7]:
turn_count = 0
MAX_TURNS = 5
# Stop words for Trigger 2 (Graceful Exit)
distress_signals = ["help", "hurry", "quick", "don't know"]

print(f"--- Session Started (Scenario: {SCENARIO}) ---")
init_res = chat.send_message(system_prompt + "\n\nStart the report session.")
print(f"\nAI: {init_res.text}\n" + "-"*50)

while True:
    user_input = input("User input (or 'EXIT'): ")
    turn_count += 1
    
    # Trigger 3: Manual Submit/Exit
    if user_input.upper() == "EXIT":
        print("\n[Trigger: Manual Exit] Finalizing report...")
        break
        
    # Trigger 2: Graceful Exit (Distress or Max Turns)
    if turn_count >= MAX_TURNS or any(word in user_input.lower() for word in distress_signals):
        print(f"\n[Trigger: Graceful Exit] Wrapping up due to turn limit or user distress...")
        final_msg = chat.send_message("The user is unable to provide more info. Stop asking and give final safety instructions.")
        print(f"\nAI: {final_msg.text}")
        break

    # Send user input to AI
    response = chat.send_message(user_input)
    
    # Trigger 1: Information Saturation (AI decides if it has enough data)
    check_query = "Internal: Do we have enough for a basic triage (species, health, location)? Reply 'READY' or 'MORE'."
    check_status = chat.send_message(check_query)
    
    if "READY" in check_status.text.upper():
        print("\n[Trigger: Information Saturation] Sufficient data collected.")
        closing = chat.send_message("Acknowledge that info is sufficient and provide final feedback.")
        print(f"\nAI: {closing.text}")
        break
    else:
        # Continue normal conversation
        print(f"\nAI: {response.text}\n" + "-"*50)

# --- 5. Final Report Generation (Feature Extraction) ---
print("\n" + "="*20 + " GENERATING TRIAGE REPORT " + "="*20)

gps_info = get_device_location() # Automatically fetch GPS data
extraction_prompt = f"""
Generate a final structured JSON report for the downstream triage model.

Use only information supported by:
1. Current CV input: {json.dumps(cv_data) if cv_data else "No CV input"}
2. The reporter conversation
3. GPS data: {json.dumps(gps_info)}

Do NOT infer or calculate severity_level, urgency_level, dispatch_priority, or final rescue decision.
The downstream triage model will calculate those.

If a field is not supported by the CV input or reporter text, set it to "unknown".
Do not invent injury, mobility, breathing status, or crowd level.

Return ONLY raw valid JSON with this schema:

{
  "incident_id": "demo_incident_001",
  "location": {
    "latitude": null,
    "longitude": null,
    "address": "",
    "source": "device_gps"
  },
  "animals": [
    {
      "animal_id": "A1",
      "species": {
        "predicted_species": "",
        "confidence": null,
        "source": "cv_model_or_reporter_text_or_unknown"
      },
      "animal_attributes": {
        "estimated_size": "unknown",
        "estimated_age_group": "unknown"
      },
      "observed_condition": {
        "injury_visible": "unknown",
        "injury_type": [],
        "mobility": "unknown",
        "responsiveness": "unknown",
        "breathing_status": "unknown",
        "eye_status": "unknown"
      },
      "environment": {
        "on_land_or_water": "unknown",
        "crowd_level": "unknown",
        "pets_nearby": "unknown",
        "accessibility": "unknown"
      },
      "feature_sources": {
        "species": "unknown",
        "injury_visible": "unknown",
        "mobility": "unknown",
        "breathing_status": "unknown"
      }
    }
  ],
  "missing_information": [],
  "reporter_guidance": [
    "Do not touch, feed, pour water on, or move the animal.",
    "Keep people and pets at a safe distance.",
    "Wait for trained responders."
  ],
  "triage_model_input": {
    "species_code": "",
    "species_confidence": null,
    "size_code": "unknown",
    "injury_visible": "unknown",
    "injury_type_codes": [],
    "mobility_code": "unknown",
    "responsiveness_code": "unknown",
    "breathing_status_code": "unknown",
    "crowd_level_code": "unknown",
    "accessibility_code": "unknown"
  }
}
"""

final_report = chat.send_message(extraction_prompt)
print(final_report.text)

--- Session Started (Scenario: BOTH) ---

AI: Okay, I see a California Sea Lion in the photo.

Please remember to stay at least 50 feet away from the animal for your safety and theirs.

Is there anything else you can tell me about what you're seeing?
--------------------------------------------------


User input (or 'EXIT'):  It is not moving



AI: Thank you for that information. So, the California Sea Lion is breathing with its eyes closed and is not moving.

Please remember to stay at least 50 feet away from the animal for your safety and theirs.

Can you tell me its exact location? For example, "on the beach near the pier," or "on rocks by the lighthouse."
--------------------------------------------------


User input (or 'EXIT'):  EXIT



[Trigger: Manual Exit] Finalizing report...

==================== GENERATING TRIAGE REPORT ====================


ResponseValidationError: The model response did not complete successfully.
Finish reason: 2.
Finish message: .
Safety ratings: [].
To protect the integrity of the chat session, the request and response were not added to chat history.
To skip the response validation, specify `model.start_chat(response_validation=False)`.
Note that letting blocked or otherwise incomplete responses into chat history might lead to future interactions being blocked by the service.

"triage_features": {
  "species": "sea_lion",
  "species_confidence": 0.90,
  "estimated_size": "medium",
  "injury_visible": true,
  "injury_type": ["bleeding", "entanglement"],
  "mobility": "low",
  "responsiveness": "weak",
  "breathing_status": "unknown",
  "on_land_or_water": "on_land",
  "crowd_level": "medium",
  "accessibility": "easy"
}